# Day 022：蒸馏基础：teacher、student、temperature 与 KL

本 Notebook 对照 `train_distillation.py`，观察 teacher/student 的角色、温度平滑、KL 蒸馏损失和 CE/KL 的 alpha 组合。

In [ ]:
import sys
from pathlib import Path
candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(root for root in candidate_roots if (root / 'minimind' / 'model' / 'model_minimind.py').exists())
sys.path.insert(0, str(repo_root / 'minimind'))
import torch
import torch.nn.functional as F
from minimind.model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from minimind.trainer.train_distillation import distillation_loss
torch.manual_seed(0)
print('repo:', repo_root)

## 1. teacher 与 student

student 接受训练；teacher 只提供参考输出，处于 eval 模式并关闭梯度。

In [ ]:
student_config = MiniMindConfig(vocab_size=20, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=1, intermediate_size=16, max_position_embeddings=32, flash_attn=False)
teacher_config = MiniMindConfig(vocab_size=20, hidden_size=8, num_hidden_layers=2, num_attention_heads=2, num_key_value_heads=1, intermediate_size=16, max_position_embeddings=32, flash_attn=False)
student = MiniMindForCausalLM(student_config)
teacher = MiniMindForCausalLM(teacher_config)
teacher.eval(); teacher.requires_grad_(False)
print('student layers:', student.config.num_hidden_layers)
print('teacher layers:', teacher.config.num_hidden_layers)
print('student training:', student.training)
print('teacher training:', teacher.training)
print('student trainable params:', sum(p.numel() for p in student.parameters() if p.requires_grad))
print('teacher trainable params:', sum(p.numel() for p in teacher.parameters() if p.requires_grad))

## 2. 同一输入的 logits

去掉最后一个时间位置后，两边都与 next-token labels 对齐；teacher 在 `no_grad()` 下没有计算图。

In [ ]:
input_ids = torch.tensor([[1, 4, 7, 2]])
student_logits = student(input_ids).logits[..., :-1, :].contiguous()
with torch.no_grad():
    teacher_logits = teacher(input_ids).logits[..., :-1, :].contiguous()
print('student shape:', tuple(student_logits.shape), 'requires_grad:', student_logits.requires_grad)
print('teacher shape:', tuple(teacher_logits.shape), 'requires_grad:', teacher_logits.requires_grad)
print('student grad_fn:', type(student_logits.grad_fn).__name__)
print('teacher grad_fn:', teacher_logits.grad_fn)

## 3. temperature 平滑

温度越高，softmax 分布越平滑，teacher 的多个候选偏好更容易传给 student。

In [ ]:
logits = torch.tensor([4.0, 2.0, 1.0])
for temperature in (1.0, 1.5, 2.0, 4.0):
    probs = F.softmax(logits / temperature, dim=-1)
    print(f'T={temperature}: {probs.tolist()}, sum={probs.sum().item():.4f}')

## 4. KL：teacher soft target 与 student 分布

`F.kl_div(student_log_probs, teacher_probs)` 在当前 API 用法下计算 `KL(teacher || student)`。分布相同时 KL 接近 0。

In [ ]:
teacher_logits = torch.tensor([[4.0, 2.0, 1.0]])
student_same = torch.tensor([[4.0, 2.0, 1.0]])
student_diff = torch.tensor([[1.0, 2.0, 4.0]])
for student_logits in (student_same, student_diff):
    print('student logits:', student_logits.tolist())
    for temperature in (1.0, 2.0):
        value = distillation_loss(student_logits, teacher_logits, temperature=temperature)
        print(f'  T={temperature}, distill loss={value.item():.6f}')

## 5. alpha 组合 CE 与 KL

`loss = alpha * ce_loss + (1-alpha) * distill_loss`。alpha 越大，越偏向真实标签 CE。

In [ ]:
ce_loss = 2.0
distill_loss_value = 0.6
for alpha in (0.0, 0.5, 0.8, 1.0):
    total = alpha * ce_loss + (1 - alpha) * distill_loss_value
    print(f'alpha={alpha}: total loss={total}')

## 6. 有效位置 mask

蒸馏只在 labels 不为 `-100` 的 next-token 位置计算，PAD 等无效位置被过滤。

In [ ]:
labels = torch.tensor([[1, 4, 7, -100]])
loss_mask = (labels[..., 1:] != -100).float()
print('loss_mask:', loss_mask)
print('flattened valid positions:', (loss_mask.view(-1) == 1).tolist())
print('logits before filter:', (2, 3, 20))
print('logits after flatten:', (2 * 3, 20))

## 今日边界

已验证 teacher/student、temperature、KL、alpha 和有效位置 mask。Day 023 进入一次真实蒸馏 forward/backward/optimizer.step。